# Preprocessing: Build a Modeling Master Dataset

This notebook reads the raw CRML CSV exports, applies deterministic record-level rules, creates a chronological train/test split, and fits a scikit-learn preprocessing pipeline using **training data only**.

### Key decisions
- Keep California residential single-family records.
- Require a valid positive `ClosePrice` and valid `CloseDate`.
- Remove records where `CloseDate` precedes `ListingContractDate`.
- Remove duplicate normalized addresses on the same close date when an address column is available.
- Hold out the latest six calendar months (rather than a random 20%).
- Drop columns with more than 80% missing values based on training data only.
- Derive the ClosePrice outlier bounds from training data and apply those bounds to both splits.
- Fit imputers, scaling, and categorical encoders on training data only.
- Export transformed train/test datasets plus a metadata file describing the target, split, and dropped columns.

**Important:** Edit `DATA_DIR` and `OUTPUT_DIR` below. Check the available columns and the address field before treating the result as final. Filtering by target/date is appropriate for a supervised closed-sale dataset; the feature exclusions should still be matched to the intended prediction time.

## 1. Imports and configuration

In [2]:
%pip install numpy<2.0.0

Note: you may need to restart the kernel to use updated packages.


The system cannot find the file specified.


In [8]:
from pathlib import Path
import json
import re
import warnings
#fix numpy version import to 2.0

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.model_selection import TimeSeriesSplit

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# EDIT THESE PATHS FOR YOUR MACHINE.
DATA_DIR = Path(r"C:\Users\Owner\Documents\IDX\CRML_Data")  # C:\Users\Owner\Documents\IDX\CRML_Data
OUTPUT_DIR = Path("./modeling_master")
TARGET = "ClosePrice"
DATE_COL = "CloseDate"
LISTING_DATE_COL = "ListingContractDate"

# Fixed domain filters described in the original notebook.
STATE_VALUE = "CA"
PROPERTY_TYPE_VALUE = "Residential"
PROPERTY_SUBTYPE_VALUE = "SingleFamilyResidence"

# Train-derived policies.
MAX_MISSING_FRACTION = 0.80
PRICE_LOWER_QUANTILE = 0.005
PRICE_UPPER_QUANTILE = 0.995
TEST_MONTHS = 6

# Candidate address fields; first matching column is used.
ADDRESS_CANDIDATES = ["UnparsedAddress", "StreetAddress", "Address", "PropertyAddress"]

## 2. Load and concatenate raw CSV exports

Read every CSV in the configured folder. This fixes the earlier issue where the concatenated dataframe was overwritten by reading only `files[0]`.

In [9]:
files = sorted(DATA_DIR.glob("*.csv"))  # reads all CSVs in the folder

df_list = [pd.read_csv(file, low_memory=False) for file in files]
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data directory does not exist: {DATA_DIR}")

if not df_list:
    raise FileNotFoundError(f"No CSV files found in: {DATA_DIR}")

raw = pd.concat(df_list, ignore_index=True)

display(raw.dtypes.astype(str).value_counts().rename_axis("dtype").to_frame("columns"))

FileNotFoundError: Data directory does not exist: C:\Users\Owner\Documents\IDX\CRML_Data

## 3. Validate columns and normalize basic types

Do not silently assume every export has every column. Required columns stop execution if missing; optional columns are handled conditionally.

In [ ]:
required = {TARGET, DATE_COL, "StateOrProvince", "PropertyType", "PropertySubType"}
missing_required = required - set(raw.columns)
if missing_required:
    raise ValueError(f"Required columns missing from the combined CSVs: {sorted(missing_required)}")

# Parse date fields. Invalid dates become NaT and are reported.
date_cols = [
    c for c in [DATE_COL, LISTING_DATE_COL, "PurchaseContractDate",
                "ContractStatusChangeDate"] if c in raw.columns
]
for col in date_cols:
    raw[col] = pd.to_datetime(raw[col], errors="coerce")

# Convert target to numeric; nonnumeric entries become missing and are removed below.
raw[TARGET] = pd.to_numeric(raw[TARGET], errors="coerce")

print("Rows:", len(raw))
print("Missing/invalid CloseDate:", int(raw[DATE_COL].isna().sum()))
print("Missing/invalid ClosePrice:", int(raw[TARGET].isna().sum()))
print("Columns:", len(raw.columns))

## 4. Apply deterministic record filters

These filters use the meaning of each record, not statistics estimated from the dataset. The filters are applied before splitting. The outlier thresholds and imputation/encoding parameters are learned only from the training set later.

In [ ]:
df = raw.copy()

df = df.loc[
    df["StateOrProvince"].astype("string").str.strip().str.upper().eq(STATE_VALUE)
    & df["PropertyType"].astype("string").str.strip().eq(PROPERTY_TYPE_VALUE)
    & df["PropertySubType"].astype("string").str.strip().eq(PROPERTY_SUBTYPE_VALUE)
].copy()

# Require valid target/date for a closed-sale modeling row.
df = df.dropna(subset=[TARGET, DATE_COL]).copy()
df = df.loc[np.isfinite(df[TARGET]) & (df[TARGET] > 0)].copy()

# Exclude close dates before listing dates where both dates are available.
if LISTING_DATE_COL in df.columns:
    invalid_date_order = (
        df[LISTING_DATE_COL].notna()
        & df[DATE_COL].notna()
        & (df[DATE_COL] < df[LISTING_DATE_COL])
    )
    print("Rows removed because CloseDate < ListingContractDate:", int(invalid_date_order.sum()))
    df = df.loc[~invalid_date_order].copy()

print("Rows after deterministic filters:", len(df))
if df.empty:
    raise ValueError("No rows remain after filters. Verify source values and filter assumptions.")

## 5. Remove duplicates

The intended rule is duplicate normalized address + same `CloseDate`, retaining the first row. If no address candidate exists, the notebook reports that and falls back to exact-row duplicates only. Review this fallback before relying on the master dataset.

In [ ]:
address_col = next((c for c in ADDRESS_CANDIDATES if c in df.columns), None)

if address_col is not None:
    normalized_address = (
        df[address_col].astype("string")
        .str.lower()
        .str.strip()
        .str.replace(r"\\s+", " ", regex=True)
    )
    # Missing addresses should not all be considered the same address.
    duplicate_key = normalized_address.notna() & normalized_address.ne("")
    duplicated_mask = pd.Series(False, index=df.index)
    duplicated_mask.loc[duplicate_key] = df.loc[duplicate_key].assign(
        _normalized_address=normalized_address.loc[duplicate_key]
    ).duplicated(subset=["_normalized_address", DATE_COL], keep="first").to_numpy()
    removed = int(duplicated_mask.sum())
    df = df.loc[~duplicated_mask].copy()
    print(f"Address/date duplicates removed using {address_col}: {removed}")
else:
    exact_duplicates = int(df.duplicated().sum())
    print(
        "WARNING: no recognized address column found. "
        f"Removing exact duplicate rows only ({exact_duplicates}); verify the address field."
    )
    df = df.drop_duplicates().copy()

df = df.sort_values(DATE_COL).reset_index(drop=True)
print("Rows after deduplication:", len(df))

## 6. Chronological train/test split

Reserve the last six calendar months relative to the latest observed close date. This preserves chronology and avoids random leakage. The split is created before learning missingness thresholds, outlier bounds, imputation values, scaling, or category levels.

In [ ]:
cutoff = df[DATE_COL].max() - pd.DateOffset(months=TEST_MONTHS)
train = df.loc[df[DATE_COL] < cutoff].copy().reset_index(drop=True)
test = df.loc[df[DATE_COL] >= cutoff].copy().reset_index(drop=True)

if train.empty or test.empty:
    raise ValueError(
        f"Empty split: train={len(train)}, test={len(test)}. "
        "Check the date coverage or change TEST_MONTHS."
    )

print("Cutoff:", cutoff)
print(f"Train rows: {len(train):,}; dates: {train[DATE_COL].min()} — {train[DATE_COL].max()}")
print(f"Test rows:  {len(test):,}; dates: {test[DATE_COL].min()} — {test[DATE_COL].max()}")
assert train[DATE_COL].max() < test[DATE_COL].min()

## 7. Learn target outlier limits from training data only

Use the stated 0.5th and 99.5th percentiles for `ClosePrice`. Apply the same frozen bounds to train and test. This filters rows based on the target, so record the number removed and understand that the model will then target the central 99% of training prices rather than all possible sale prices.

In [ ]:
price_lower = float(train[TARGET].quantile(PRICE_LOWER_QUANTILE))
price_upper = float(train[TARGET].quantile(PRICE_UPPER_QUANTILE))
if not np.isfinite(price_lower) or not np.isfinite(price_upper) or price_lower > price_upper:
    raise ValueError(f"Invalid price bounds: {price_lower}, {price_upper}")

def filter_price_bounds(frame, lower, upper, target=TARGET):
    mask = frame[target].between(lower, upper, inclusive="both")
    return frame.loc[mask].copy(), int((~mask).sum())

train, n_train_removed = filter_price_bounds(train, price_lower, price_upper)
test, n_test_removed = filter_price_bounds(test, price_lower, price_upper)

print(f"Training price bounds: [{price_lower:,.2f}, {price_upper:,.2f}]")
print(f"Train rows removed: {n_train_removed:,}")
print(f"Test rows removed using training bounds: {n_test_removed:,}")
print(f"Rows retained — train: {len(train):,}; test: {len(test):,}")

if train.empty or test.empty:
    raise ValueError("Price filtering created an empty split. Review the quantiles and data.")

## 8. Drop columns with excessive missingness using training data

The original notes specify dropping columns with more than 80% missingness. Calculate this threshold on training data only, then apply the same feature list to test. The target is retained separately and `_source_file` is removed because it is ingestion metadata, not a property feature.

In [ ]:
metadata_cols = ["_source_file"]
protected_cols = [TARGET, DATE_COL]
candidate_feature_cols = [
    c for c in train.columns
    if c not in metadata_cols and c not in protected_cols
]

missing_fraction_train = train[candidate_feature_cols].isna().mean()
dropped_missing_cols = missing_fraction_train[
    missing_fraction_train > MAX_MISSING_FRACTION
].index.tolist()

print(f"Columns dropped (> {MAX_MISSING_FRACTION:.0%} missing in train):", dropped_missing_cols)

feature_cols = [c for c in candidate_feature_cols if c not in dropped_missing_cols]
X_train_raw = train[feature_cols].copy()
X_test_raw = test.reindex(columns=feature_cols).copy()
y_train = train[TARGET].copy()
y_test = test[TARGET].copy()

print("Raw feature shapes:", X_train_raw.shape, X_test_raw.shape)

## 9. Feature engineering and data-quality rules

This function is deterministic and is applied identically to train and test. Statistical imputation is handled later inside the scikit-learn pipeline, where fit is called only on training data.

In [ ]:
def engineer_features(X):
    X = X.copy()

    # Date-derived features: retain numeric month/year features, not raw dates.
    for col in ["ListingContractDate", "PurchaseContractDate", "ContractStatusChangeDate"]:
        if col in X.columns:
            parsed = pd.to_datetime(X[col], errors="coerce")
            prefix = col.replace("Date", "")
            X[f"{prefix}_month"] = parsed.dt.month.astype("float64")
            X[f"{prefix}_year"] = parsed.dt.year.astype("float64")
            X = X.drop(columns=[col])

    # Building age is relative to each property's close date when possible.
    if "YearBuilt" in X.columns:
        year_built = pd.to_numeric(X["YearBuilt"], errors="coerce")
        if DATE_COL in X.columns:
            close_year = pd.to_datetime(X[DATE_COL], errors="coerce").dt.year
        else:
            close_year = pd.Series(np.nan, index=X.index)
        X["BuildingAge"] = close_year - year_built
        # Invalid ages become missing and will be imputed using training median.
        X.loc[X["BuildingAge"] < 0, "BuildingAge"] = np.nan
        X = X.drop(columns=["YearBuilt"])

    # Standardize postal codes to 5-digit strings when possible.
    if "PostalCode" in X.columns:
        def normalize_postal(value):
            if pd.isna(value):
                return np.nan
            s = str(value).strip()
            # Handle CSV-imported values such as 92101.0.
            if re.fullmatch(r"\\d+\\.0", s):
                s = s[:-2]
            digits = re.sub(r"\\D", "", s)
            if len(digits) == 5:
                return digits
            if len(digits) == 9:
                return digits[:5]
            if len(s) == 5 and s.isdigit():
                return s
            return np.nan
        X["PostalCode"] = X["PostalCode"].map(normalize_postal).astype("object")

    # Coordinates outside broad California bounds are marked missing.
    # These are intentionally broad plausibility bounds, not a precise state polygon.
    if "Latitude" in X.columns:
        lat = pd.to_numeric(X["Latitude"], errors="coerce")
        X["Latitude"] = lat.where(lat.between(32.0, 42.5))
    if "Longitude" in X.columns:
        lon = pd.to_numeric(X["Longitude"], errors="coerce")
        X["Longitude"] = lon.where(lon.between(-125.0, -113.0))

    # Prefer square feet as the standard lot-size unit.
    if "LotSizeSquareFeet" not in X.columns and "LotSizeAcres" in X.columns:
        acres = pd.to_numeric(X["LotSizeAcres"], errors="coerce")
        X["LotSizeSquareFeet"] = acres * 43560
    elif "LotSizeSquareFeet" in X.columns and "LotSizeAcres" in X.columns:
        sqft = pd.to_numeric(X["LotSizeSquareFeet"], errors="coerce")
        acres = pd.to_numeric(X["LotSizeAcres"], errors="coerce")
        X["LotSizeSquareFeet"] = sqft.where(sqft > 0, acres * 43560)
        X = X.drop(columns=["LotSizeAcres"])
    elif "LotSizeSquareFeet" in X.columns:
        X["LotSizeSquareFeet"] = pd.to_numeric(X["LotSizeSquareFeet"], errors="coerce")

    # Invalid physical measurements become missing; the pipeline imputes them.
    positive_cols = [
        "LivingArea", "LotSizeSquareFeet", "LotSizeArea",
        "LotSizeAcres", "AssociationFee", "TaxAnnualAmount"
    ]
    for col in positive_cols:
        if col in X.columns:
            values = pd.to_numeric(X[col], errors="coerce")
            X[col] = values.where(values > 0)

    # Nonnegative count/size fields: negative values are invalid.
    nonnegative_cols = [
        "BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces",
        "ParkingTotal", "Stories"
    ]
    for col in nonnegative_cols:
        if col in X.columns:
            values = pd.to_numeric(X[col], errors="coerce")
            X[col] = values.where(values >= 0)

    # Remove raw close date from predictors; it is used only for splitting/feature engineering.
    if DATE_COL in X.columns:
        X = X.drop(columns=[DATE_COL])

    return X

X_train_eng = engineer_features(X_train_raw)
X_test_eng = engineer_features(X_test_raw)

# Ensure identical feature names and ordering.
X_test_eng = X_test_eng.reindex(columns=X_train_eng.columns)
print("Engineered shapes:", X_train_eng.shape, X_test_eng.shape)
display(X_train_eng.head())

## 10. Create missingness indicators for binary features

For the known yes/no flags, add a missingness indicator before imputation. Original source encodings vary across exports, so normalize common yes/no and boolean forms. The imputation mode is learned in the pipeline from training rows only.

In [ ]:
BINARY_CANDIDATES = [
    "ViewYN", "PoolPrivateYN", "FireplaceYN", "NewConstructionYN"
]

def normalize_binary_columns(X):
    X = X.copy()
    true_values = {"y", "yes", "true", "1", "1.0"}
    false_values = {"n", "no", "false", "0", "0.0"}

    for col in BINARY_CANDIDATES:
        if col not in X.columns:
            continue
        raw = X[col]
        text = raw.astype("string").str.strip().str.lower()
        normalized = pd.Series(np.nan, index=X.index, dtype="object")
        normalized.loc[text.isin(true_values)] = "Yes"
        normalized.loc[text.isin(false_values)] = "No"
        # Preserve already meaningful categories only if they match expected values.
        normalized.loc[raw.isna()] = np.nan
        X[f"{col}_was_missing"] = raw.isna().astype("int8")
        X[col] = normalized
    return X

X_train_eng = normalize_binary_columns(X_train_eng)
X_test_eng = normalize_binary_columns(X_test_eng)
X_test_eng = X_test_eng.reindex(columns=X_train_eng.columns)

print("Binary columns found:", [c for c in BINARY_CANDIDATES if c in X_train_eng.columns])

## 11. Remove columns that cannot be used reliably

Some raw exports may contain constant columns, mixed-type columns, or all-missing columns after feature engineering. Determine these columns from training data only and apply the same drop list to test.

In [ ]:
all_missing_cols = X_train_eng.columns[X_train_eng.isna().all()].tolist()
constant_cols = [
    c for c in X_train_eng.columns
    if c not in all_missing_cols and X_train_eng[c].nunique(dropna=True) <= 1
]
drop_after_engineering = sorted(set(all_missing_cols + constant_cols))
print("All-missing columns:", all_missing_cols)
print("Constant columns:", constant_cols)

X_train_eng = X_train_eng.drop(columns=drop_after_engineering, errors="ignore")
X_test_eng = X_test_eng.drop(columns=drop_after_engineering, errors="ignore")
X_test_eng = X_test_eng.reindex(columns=X_train_eng.columns)

## 12. Build the scikit-learn preprocessing pipeline

Numeric columns: median imputation + standard scaling. Categorical columns: most-frequent imputation + one-hot encoding with unknown categories ignored. All imputers, scaling parameters, and category vocabularies are fitted on training data only.

This pipeline is for the predictors; target transformation is deliberately kept separate so evaluation can be reported in original price units.

In [ ]:
numeric_features = X_train_eng.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X_train_eng.columns if c not in numeric_features]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=False)),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

transformers = []
if numeric_features:
    transformers.append(("numeric", numeric_pipeline, numeric_features))
if categorical_features:
    transformers.append(("categorical", categorical_pipeline, categorical_features))

if not transformers:
    raise ValueError("No usable features remain after filtering and feature engineering.")

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    verbose_feature_names_out=False
)

X_train_array = preprocessor.fit_transform(X_train_eng)
X_test_array = preprocessor.transform(X_test_eng)

try:
    feature_names_out = preprocessor.get_feature_names_out()
except AttributeError:
    feature_names_out = [f"feature_{i}" for i in range(X_train_array.shape[1])]

X_train_processed = pd.DataFrame(
    X_train_array, columns=feature_names_out, index=X_train_eng.index
)
X_test_processed = pd.DataFrame(
    X_test_array, columns=feature_names_out, index=X_test_eng.index
)

print("Processed train:", X_train_processed.shape)
print("Processed test:", X_test_processed.shape)
assert X_train_processed.columns.equals(X_test_processed.columns)
assert np.isfinite(X_train_processed.to_numpy(dtype=float)).all()
assert np.isfinite(X_test_processed.to_numpy(dtype=float)).all()

## 13. Create the master datasets

The modeling master dataset contains processed features and the original `ClosePrice` target. Also save untransformed targets separately. The row index is preserved only as an export index; it is not a predictor.

In [ ]:
master_train = X_train_processed.copy()
master_train[TARGET] = y_train.reset_index(drop=True).to_numpy()
master_train[DATE_COL] = train[DATE_COL].reset_index(drop=True).to_numpy()

master_test = X_test_processed.copy()
master_test[TARGET] = y_test.reset_index(drop=True).to_numpy()
master_test[DATE_COL] = test[DATE_COL].reset_index(drop=True).to_numpy()

print("Master train shape:", master_train.shape)
print("Master test shape:", master_test.shape)
display(master_train.head())

## 14. Optional log target for model comparison

Keep the original target. Use `log1p(ClosePrice)` only as an alternative training target for models where it is useful; invert predictions with `expm1` before evaluating in dollars.

In [ ]:
y_train_log = np.log1p(y_train.to_numpy(dtype=float))
y_test_log = np.log1p(y_test.to_numpy(dtype=float))

print("Original target median:", float(y_train.median()))
print("Log target median:", float(np.median(y_train_log)))

## 15. Export datasets and preprocessing artifacts

The fitted preprocessor and feature schema are needed to transform future data consistently. Pickle/joblib artifacts should only be loaded from trusted sources and within a compatible Python/scikit-learn environment.

In [ ]:
import joblib

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

master_train.to_csv(OUTPUT_DIR / "master_train.csv", index=False)
master_test.to_csv(OUTPUT_DIR / "master_test.csv", index=False)
pd.DataFrame({"ClosePrice": y_train.to_numpy()}).to_csv(
    OUTPUT_DIR / "y_train.csv", index=False
)
pd.DataFrame({"ClosePrice": y_test.to_numpy()}).to_csv(
    OUTPUT_DIR / "y_test.csv", index=False
)
joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")

metadata = {
    "target": TARGET,
    "date_column": DATE_COL,
    "cutoff": str(cutoff),
    "test_months": TEST_MONTHS,
    "train_rows_after_filtering": int(len(train)),
    "test_rows_after_filtering": int(len(test)),
    "price_lower_bound_from_train": price_lower,
    "price_upper_bound_from_train": price_upper,
    "max_missing_fraction": MAX_MISSING_FRACTION,
    "dropped_missing_columns": dropped_missing_cols,
    "dropped_after_engineering": drop_after_engineering,
    "address_column_used": address_col,
    "numeric_features_before_encoding": numeric_features,
    "categorical_features_before_encoding": categorical_features,
    "processed_feature_names": list(map(str, feature_names_out)),
}
with open(OUTPUT_DIR / "preprocessing_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved outputs to:", OUTPUT_DIR.resolve())
print(sorted(p.name for p in OUTPUT_DIR.iterdir()))

## 16. Final audit

Review the counts and exclusions before modeling. Do not interpret the filtered master dataset as representing all California residential sales: it represents records meeting the stated filters and the training-derived target range.

In [ ]:
audit = pd.DataFrame({
    "dataset": ["train", "test"],
    "rows": [len(master_train), len(master_test)],
    "min_close_date": [train[DATE_COL].min(), test[DATE_COL].min()],
    "max_close_date": [train[DATE_COL].max(), test[DATE_COL].max()],
    "min_close_price": [y_train.min(), y_test.min()],
    "median_close_price": [y_train.median(), y_test.median()],
    "max_close_price": [y_train.max(), y_test.max()],
})
display(audit)
print("Feature columns aligned:", X_train_processed.columns.equals(X_test_processed.columns))
print("Any missing processed train values:", bool(X_train_processed.isna().any().any()))
print("Any missing processed test values:", bool(X_test_processed.isna().any().any()))
print("Any non-finite processed train values:", not bool(np.isfinite(X_train_processed.to_numpy(dtype=float)).all()))
print("Any non-finite processed test values:", not bool(np.isfinite(X_test_processed.to_numpy(dtype=float)).all()))